# 重回帰分析

現実の予測問題では、出力は 1 つの理由だけで決まりません。面積、駅距離、築年数のように、複数の事情が同時に効いている状況を扱うのが重回帰です。

重回帰は単回帰の延長にありますが、初学者が最初に迷いやすいのは『入力が増えると何が良くなり、何が難しくなるのか』です。ここでは、式を先に暗記するより、複数の入力を同時に見る必要性と、係数の読み方が急に難しくなる理由を順に押さえます。

## まずは『1つの入力では足りない』場面を考える

たとえば家賃を予測したいとき、面積だけを見れば十分でしょうか。実際には、駅からの距離や築年数、部屋の向きなど、複数の事情が同時に効いています。1つの入力だけで説明しようとすると、見落とした要因のぶんまで無理やりその1本の係数に背負わせることになります。

重回帰は、その『複数の事情を同時に見たい』という素朴な要求に答えるための最小モデルです。単回帰で学んだ予測・誤差・評価の流れはそのままに、入力を複数に広げたものだと考えると入りやすくなります。

## 複数の事情が同時に効くと何が難しくなるか

入力が複数になると、予測の精度だけでなく『どの入力がどれくらい効いたのか』の切り分けが問題になります。`x1` が効いているように見えても、実際には `x2` と似た動きをしているだけかもしれません。

だから重回帰では、係数の値そのものだけでなく、その値をどこまで安心して読めるかも大事になります。ここが単回帰より一段難しくなる点です。

複数要因を含む疑似データを作り、それを 1 本の式へまとめ、係数の安定性を確かめます。計算そのものより、係数がどの条件で素直に読めて、どの条件で怪しくなるかが中心になります。

## 最初に知っておきたい言葉

- 重回帰: 複数の入力特徴量を同時に使う回帰
- 切片: すべての入力が 0 のときの基準値
- 係数: 他の入力を固定したとき、その特徴量が 1 増えると予測がどれだけ変わるか
- 多重共線性: 入力どうしが似すぎていて、係数の解釈が不安定になる状態

最初は厳密な定義を暗記する必要はありません。『入力を複数にしたとき、どの入力の影響かを切り分けにくくなることがある』という問題意識が持てれば十分です。

## 係数を出して終わりにはならない

重回帰の難しさは、係数が求まったあとに始まります。数値が出たという事実と、その数値を寄与として読んでよいという判断は別物です。

## 出力は『どちら向きに効いたか』を見る入口

`beta_hat` は各特徴量の寄与を表す推定値です。正なら増やす方向、負なら減らす方向に効くと読めます。

ただし、その値が少しデータを変えただけで大きく揺れるなら、解釈の土台は弱いままです。重回帰では『数値が出た』だけで安心しないことが重要です。

## 行列の式がしていること

`(X^T X)^(-1) X^T y` という形は威圧的に見えますが、やっていることは全データをまとめて見て、残差がいちばん小さくなる係数の組を探すことです。

見るべき本質は、式の長さよりも『各入力がどれだけ別々の情報を持っているか』にあります。入力どうしが似すぎると、数式は解けても解釈が不安定になります。

## 扱う範囲

最小二乗解と多重共線性の入口に焦点を当てます。変数選択や正則化は先へ譲り、複数特徴量を同時に読むと何が変わるのかを明確にします。

## 複数要因を含むデータ

`x1` と `x2` の両方が `y` に効く疑似データを使います。片方だけを見ていたのでは、もう片方の寄与を取り違えやすい状況です。

In [ ]:
import numpy as np
np.random.seed(11)

n = 120
x1 = np.random.randn(n)
x2 = 0.6 * x1 + 0.8 * np.random.randn(n)  # 相関あり
eps = 0.5 * np.random.randn(n)
y = 1.2 + 2.0 * x1 - 1.5 * x2 + eps

X = np.column_stack([np.ones(n), x1, x2])


## まとめて解くと係数はどう決まるか

単回帰では傾きと切片を手で追えましたが、変数が増えると一括で解く方が自然になります。先頭に入る 1 の列は切片のための列で、直線を原点へ固定しない役割を持ちます。

ここでも大事なのは、数式の形より『複数の入力をまとめて見て、いちばん外れにくい組み合わせを探している』という読み方です。

In [ ]:
# 最小二乗解: beta = (X^T X)^(-1) X^T y
beta = np.linalg.pinv(X.T @ X) @ X.T @ y
pred = X @ beta
mse = np.mean((pred - y) ** 2)

print('beta_hat =', np.round(beta, 6))
print('train MSE=', round(mse, 6))


## 入力どうしが似すぎると何が起きるか

VIF を見る理由は、予測精度の確認ではなく、係数をどこまで信用して読めるかを確かめるためです。説明変数が似た情報を持ちすぎると、どちらが効いたのかを切り分けにくくなります。

初学者向けに言い換えるなら、『似たものを2回数えているような状態だと、どちらの係数に役割を割り振るべきか不安定になる』ということです。

In [ ]:
# VIFで多重共線性を確認
# VIF_j = 1 / (1 - R_j^2)

def vif(x_target, x_other):
    Xo = np.column_stack([np.ones(len(x_other)), x_other])
    b = np.linalg.pinv(Xo.T @ Xo) @ Xo.T @ x_target
    pred = Xo @ b
    ss_res = np.sum((x_target - pred) ** 2)
    ss_tot = np.sum((x_target - x_target.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot
    return 1.0 / (1.0 - r2)

vif_x1 = vif(x1, x2)
vif_x2 = vif(x2, x1)
print('VIF(x1)=', round(vif_x1, 6), 'VIF(x2)=', round(vif_x2, 6))


重回帰では、係数推定と同じくらい係数の安定性が重要です。説明モデルとして使うなら、値の大小だけでなく、その値がどの程度揺れやすいかまで読む必要があります。単回帰の延長ではありますが、『複数入力にした瞬間、解釈には新しい注意点が増える』ことをここで押さえてください。